In [1]:
pip install transformers datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [1]:
from datasets import load_dataset

dataset = load_dataset("zillow/real_estate_v1")["train"]

In [2]:
def build_qa_pairs(example):
    pairs = []
    messages = example["messages"]
    
    for i in range(len(messages) - 1):
        if messages[i]["role"] == "user" and messages[i+1]["role"] == "assistant":
            pairs.append({
                "input_text": messages[i]["content"],
                "target_text": messages[i+1]["content"]
            })
    
    return {"pairs": pairs}

In [27]:
qa_data = []

for example in dataset:
    messages = example["messages"]
    for i in range(len(messages) - 1):
        if messages[i]["role"] == "user" and messages[i+1]["role"] == "assistant":
            qa_data.append({
                "input_text": messages[i]["content"],
                "target_text": messages[i+1]["content"]
            })

from datasets import Dataset
qa_dataset = Dataset.from_list(qa_data)

print(len(qa_dataset))

29440


In [28]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [29]:
import re

def clean_text(text):
    text = re.sub(r"\*+", "", text)      # remove bold
    text = re.sub(r"#+", "", text)       # remove markdown headers
    text = re.sub(r"\s+", " ", text)     # normalize spaces
    return text.strip()

In [30]:
def clean_example(example):
    example["input_text"] = clean_text(example["input_text"])
    example["target_text"] = clean_text(example["target_text"])
    return example

qa_dataset = qa_dataset.map(clean_example)

Map:   0%|          | 0/29440 [00:00<?, ? examples/s]

In [31]:
def tokenize(example):
    input_texts = ["answer question: " + text for text in example["input_text"]]

    model_inputs = tokenizer(
        input_texts,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    labels = tokenizer(
        example["target_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    
    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized = qa_dataset.map(tokenize, batched=True)
tokenized.set_format("torch")

Map:   0%|          | 0/29440 [00:00<?, ? examples/s]

In [32]:
split = tokenized.train_test_split(test_size=0.1)

train_dataset = split["train"]
eval_dataset = split["test"]

In [33]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./t5_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

c:\Users\viole\anaconda3\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.917900,2.609124
2,2.791800,2.510698
3,2.747800,2.485169


TrainOutput(global_step=9936, training_loss=2.8980235838467756, metrics={'train_runtime': 2778.079, 'train_samples_per_second': 28.613, 'train_steps_per_second': 3.577, 'total_flos': 5379024557703168.0, 'train_loss': 2.8980235838467756, 'epoch': 3.0})

In [39]:
question = "I’ve been hearing a lot about mortgage contingencies in purchase agreements. Can you explain what they are and why they’re important?"

input_text = "Provide a clear and accurate explanation to the following real estate question:\n\n" + question

device = model.device

inputs = tokenizer(input_text, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The mortgage contingency in purchase agreements is a significant factor in buying a property. This means that you are required to negotiate with the lender before making any major changes. Here are some key points to consider: 1. Mortgage Contingencies: - Mortgage Contending: A good deal can lead to higher mortgage rates, which can lower your mortgage rate. This can be due to lower interest rates and lower monthly payments. - Incentives: There are several different types of mortgage contidings that can affect your home’s value. 2. Loan Financing: Ensure you have a clear understanding of the loan amount. – The lender will advise you to review the contract for a better price. This includes borrowing costs, taxes, and any other expenses. 3. Mortgage Contantencies: If you agree to make a sale within the initial timeframe, they may require more substantial savings to cover the cost of mortgage loans. This could include repairs or repairs to


In [36]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

Total parameters: 60,506,624


In [37]:
model.save_pretrained("my_t5_model")

In [38]:
import os

total_size = 0
for dirpath, dirnames, filenames in os.walk("my_t5_model"):
    for f in filenames:
        fp = os.path.join(dirpath, f)
        total_size += os.path.getsize(fp)

print(f"Model size: {total_size / (1024*1024):.2f} MB")

Model size: 230.83 MB


In [43]:
!pip install tensorflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

langchain-core 0.3.56 requires packaging<25,>=23.2, but you have packaging 25.0 which is incompatible.

streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.


streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompatible.


streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.0 which is incompatible.


  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/331.9 MB 3.4 MB/s eta 0:01:39
   ---------------------------------------- 1.3/331.9 MB 3.9 MB/s eta 0:01:24
   ---------------------------------------- 2.9/331.9 MB 5.2 MB/s eta 0:01:03
    --------------------------------------- 5.2/331.9 MB 6.8 MB/s eta 0:00:49
    --------------------------------------- 8.1/331.9 MB 8.5 MB/s eta 0:00:38
   - -------------------------------------- 11.3/331.9 MB 9.6 MB/s eta 0:00:34
   - -------------------------------------- 16.0/331.9 MB 11.6 MB/s eta 0:00:28
   -- ------------------------------------- 22.3/331.9 MB 14.1 MB/s eta 0:00:22
   --- ------------------------------------ 29.9/331.9 MB 16.6 MB/s eta 0:00:19
   ---- ----------------------------------- 37.2/331.9 MB 18.6 MB/s eta 0:00:16
   ----- ---------------------------------- 44.8/331.9 MB 20.2 M

In [3]:
from transformers import TFT5ForConditionalGeneration

tf_model = TFT5ForConditionalGeneration.from_pretrained(
    "my_t5_model",
    from_pt=True 
)

All PyTorch model weights were used when initializing TFT5ForConditionalGeneration.

All the weights of TFT5ForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


In [4]:
tf_model.save_pretrained("my_t5_tf", saved_model=True)

INFO:tensorflow:Assets written to: my_t5_tf\saved_model\1\assets


INFO:tensorflow:Assets written to: my_t5_tf\saved_model\1\assets
